# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the [FAIR^2 Crowdsourced Data Package](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"\nPublished by: {getattr(metadata, 'author', '--')}")
print(f"Version: {getattr(metadata, 'version', '--')}")

## 2. Data Overview

Review available record sets, fields, and their IDs.

The dataset may include multiple record sets. We'll list their `@id`s, names, and the fields in each record set.

In [ ]:
# Explore the available record sets
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in metadata. The dataset may expose tables through DataFiles.")
else:
    for rs in record_sets:
        print(f"\nRecordSet '@id': {rs.id}")
        print(f"  Name: {getattr(rs, 'name', '--')}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - '@id': {field.id} | Name: {getattr(field, 'name', '--')}")

## 3. Data Extraction

Load data from available record sets. Use the record set and field `@id`s from the overview above. If there are multiple record sets, this will extract all of them into dataframes. If none are found, check for direct files via `dataset.files`.

In [ ]:
dataframes = dict()

if record_sets:
    record_set_ids = [rs.id for rs in record_sets]
    for rs_id in record_set_ids:
        print(f"Reading records from record set '@id': {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            print(f" - Loaded {len(df)} records. Columns: {df.columns.tolist()[:6]}... (+more if present)")
            dataframes[rs_id] = df
        else:
            print(f" - No records found in {rs_id}.")
else:
    print("Attempting to extract DataFrames from DataFile distributions (no record sets annotated).\n")
    file_objs = list(dataset.files)
    for fo in file_objs:
        print(f"File '@id': {fo.id}")
        try:
            df = pd.DataFrame(list(dataset.records(file_object=fo.id)))
            print(f" - Loaded {len(df)} records. Columns: {df.columns.tolist()[:6]}... (+more if present)")
            dataframes[fo.id] = df
        except Exception as e:
            print(f" - Could not load records from file: {e}")

# Show example DataFrame keys and a preview
if dataframes:
    first_key = next(iter(dataframes.keys()))
    print(f"\nLoaded dataframes: {list(dataframes.keys())}\n")
    print(f"Example columns from '{first_key}': {dataframes[first_key].columns.tolist()}\n")
    display(dataframes[first_key].head())
else:
    print("No tables could be loaded from this dataset.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes.

You should determine numeric fields from the available DataFrame.

In [ ]:
import numpy as np

# Pick the first loaded dataframe for demonstration
if dataframes:
    record_set_id = first_key
    df = dataframes[record_set_id]
    
    # Attempt to automatically select a numeric field for filtering/normalizing
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        # Try to convert columns that look numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                pass
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Using numeric field '@id'/name: '{numeric_field}' for EDA example.")
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        # Filter
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (mean value):\n", filtered_df.head())
        
        # Normalize
        normalized_name = f"{numeric_field}_normalized"
        filtered_df[normalized_name] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, normalized_name]].head())
        # Group by non-numeric field if possible
        group_candidates = [c for c in df.columns if c not in numeric_cols]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"\nGrouping data by '{group_field}':")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No categorical/group field found for grouping.")
    else:
        print("No numeric fields available for filtering or normalization.")
else:
    print("No data available for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_cols:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    if "group_field" in locals() and group_field:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=90)
        plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion

In this notebook, you loaded the [FAIR^2 Mass Spectrometry EV dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using `mlcroissant`, explored available data structures, and performed basic EDA and visualization. This workflow can be extended to more in-depth analysis tasks such as identifying protein abundance patterns, modification analysis, and more, always referencing dataset structures by their Croissant `@id`s for consistency and reproducibility.